# 09 - FEA CSV Workflow

Load FEA stress results from a CSV file, compute derived quantities
(von Mises, principal stresses, max shear), and perform hot-spot extraction.

In [ ]:
import numpy as np
import pandas as pd
from weldfatigue.fea.generic_reader import GenericCSVReader
from weldfatigue.fea.stress_tensor import StressTensorOps
from weldfatigue.fea.hotspot_extractor import HotSpotExtractor

## Create Synthetic FEA Data

In [ ]:
# Simulate stress field near a weld toe
n_nodes = 50
x = np.linspace(0, 50, n_nodes)

# Stress decays away from weld toe (x=0)
sigma_xx = 200 * np.exp(-0.04 * x)
sigma_yy = 80 * np.exp(-0.03 * x)
sigma_zz = np.zeros(n_nodes)
tau_xy = 30 * np.exp(-0.05 * x)
tau_yz = np.zeros(n_nodes)
tau_xz = np.zeros(n_nodes)

df = pd.DataFrame({
    'node_id': np.arange(1, n_nodes + 1),
    'x': x, 'y': np.zeros(n_nodes), 'z': np.zeros(n_nodes),
    'sigma_xx': sigma_xx, 'sigma_yy': sigma_yy, 'sigma_zz': sigma_zz,
    'tau_xy': tau_xy, 'tau_yz': tau_yz, 'tau_xz': tau_xz,
})
print(f"Synthetic data: {n_nodes} nodes along x-axis")
df.head(10)

## Load into FEAResult

In [ ]:
reader = GenericCSVReader()
fea = reader.read_dataframe(df)
print(f"Nodes: {fea.n_nodes}")
print(f"Solver: {fea.source_solver}")
print(f"Fields: {list(fea.nodal_fields.keys())}")

## Compute Derived Stress Quantities

In [ ]:
stress_tensor = fea.get_stress_tensor()

vm = StressTensorOps.von_mises(stress_tensor)
principals = StressTensorOps.principal_stresses(stress_tensor)
max_shear = StressTensorOps.max_shear(stress_tensor)
hydro = StressTensorOps.hydrostatic(stress_tensor)

print(f"Von Mises range: {vm.min():.1f} – {vm.max():.1f} MPa")
print(f"Max principal:   {principals[:, 0].max():.1f} MPa")
print(f"Max shear:       {max_shear.max():.1f} MPa")

## Plot Stress Distribution Along Path

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x, vm, 'b-', linewidth=2, label='Von Mises')
ax.plot(x, principals[:, 0], 'r--', linewidth=2, label='σ₁ (max principal)')
ax.plot(x, max_shear, 'g-.', linewidth=2, label='Max shear')
ax.plot(x, hydro, 'm:', linewidth=2, label='Hydrostatic')

ax.set_xlabel('Distance from weld toe [mm]', fontsize=12)
ax.set_ylabel('Stress [MPa]', fontsize=12)
ax.set_title('Stress Distribution Along Surface Path', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Hot-Spot Stress Extraction

In [ ]:
t = 10.0  # plate thickness [mm]
extractor = HotSpotExtractor(fea, plate_thickness=t)

hs_stress = extractor.compute_hotspot_stress(
    weld_toe_node=1,
    path_direction=np.array([1.0, 0.0, 0.0]),
    hotspot_type='a',
)
print(f"Hot-spot stress (Type a): {hs_stress:.2f} MPa")

hs_b = extractor.compute_hotspot_stress(
    weld_toe_node=1,
    path_direction=np.array([1.0, 0.0, 0.0]),
    hotspot_type='b',
)
print(f"Hot-spot stress (Type b): {hs_b:.2f} MPa")